# OLTP vs OLAP: Sistemas Transaccionales vs Analíticos

## Objetivos de Aprendizaje

Al finalizar este notebook, serás capaz de:
- Distinguir entre sistemas OLTP y OLAP
- Identificar características y casos de uso de cada tipo
- Comprender las diferencias arquitectónicas
- Seleccionar el tipo de sistema apropiado según el contexto

## 1. OLTP - Online Transaction Processing

### Definición

**OLTP** son sistemas diseñados para gestionar y procesar **transacciones operacionales** en tiempo real.

### Características Principales

- **Operaciones:** INSERT, UPDATE, DELETE (escrituras frecuentes)
- **Volumen:** Muchas transacciones pequeñas y rápidas
- **Usuarios:** Cientos o miles de usuarios concurrentes
- **Datos:** Actualizados en tiempo real
- **Diseño:** Normalizado (3FN) para evitar redundancia
- **Objetivo:** Integridad y consistencia de datos

### Ejemplos de Sistemas OLTP

- 🏦 **Banca:** Transferencias, retiros, depósitos
- 🛒 **E-commerce:** Procesamiento de pedidos, gestión de inventario
- ✈️ **Reservas:** Vuelos, hoteles, eventos
- 🏥 **Salud:** Registro de pacientes, citas médicas
- 📱 **CRM:** Gestión de clientes, leads, oportunidades

In [ ]:
# Simulación de una base de datos OLTP
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Ejemplo: Sistema OLTP de una tienda online
np.random.seed(42)

# Tabla Clientes (normalizada)
clientes = pd.DataFrame({
    'cliente_id': range(1, 6),
    'nombre': ['Ana García', 'Luis Pérez', 'María López', 'Carlos Ruiz', 'Laura Martín'],
    'email': ['ana@email.com', 'luis@email.com', 'maria@email.com', 'carlos@email.com', 'laura@email.com'],
    'fecha_registro': pd.date_range(start='2024-01-01', periods=5, freq='15D')
})

# Tabla Productos (normalizada)
productos = pd.DataFrame({
    'producto_id': range(1, 6),
    'nombre': ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Webcam'],
    'precio': [899.99, 29.99, 79.99, 299.99, 59.99],
    'categoria_id': [1, 2, 2, 1, 2]
})

# Tabla Pedidos (transacciones)
pedidos = pd.DataFrame({
    'pedido_id': range(1, 11),
    'cliente_id': np.random.randint(1, 6, 10),
    'producto_id': np.random.randint(1, 6, 10),
    'cantidad': np.random.randint(1, 4, 10),
    'fecha_pedido': pd.date_range(start='2024-10-01', periods=10, freq='2D'),
    'estado': np.random.choice(['Pendiente', 'Procesando', 'Enviado', 'Entregado'], 10)
})

print("="*60)
print("EJEMPLO DE BASE DE DATOS OLTP (Normalizada)")
print("="*60)
print("\n📋 Tabla CLIENTES:")
print(clientes)
print("\n📦 Tabla PRODUCTOS:")
print(productos)
print("\n🛍️ Tabla PEDIDOS (Transacciones):")
print(pedidos)

## 2. OLAP - Online Analytical Processing

### Definición

**OLAP** son sistemas diseñados para **análisis multidimensional** y consultas complejas sobre grandes volúmenes de datos históricos.

### Características Principales

- **Operaciones:** SELECT (lecturas complejas)
- **Volumen:** Consultas grandes sobre datos históricos
- **Usuarios:** Decenas de analistas y científicos de datos
- **Datos:** Históricos, consolidados
- **Diseño:** Desnormalizado (estrella/copo de nieve) para optimizar consultas
- **Objetivo:** Velocidad de consulta y análisis

### Ejemplos de Uso OLAP

- 📊 **BI:** Dashboards ejecutivos, reportes de ventas
- 📈 **Análisis de tendencias:** Comportamiento de clientes
- 🔍 **Data Mining:** Patrones y correlaciones
- 🎯 **Forecasting:** Predicciones y planificación
- 📉 **KPIs:** Indicadores de desempeño

In [ ]:
# Simulación de un Data Warehouse OLAP (desnormalizado)
# Modelo en estrella: tabla de hechos + dimensiones

# Crear tabla de hechos desnormalizada (combinando todas las tablas OLTP)
hechos_ventas = pedidos.merge(clientes[['cliente_id', 'nombre']], on='cliente_id') \
                       .merge(productos[['producto_id', 'nombre', 'precio']], on='producto_id')

hechos_ventas.columns = ['pedido_id', 'cliente_id', 'producto_id', 'cantidad', 
                         'fecha_pedido', 'estado', 'nombre_cliente', 'nombre_producto', 'precio_unitario']

# Calcular métricas agregadas
hechos_ventas['ingreso_total'] = hechos_ventas['cantidad'] * hechos_ventas['precio_unitario']

print("\n" + "="*80)
print("EJEMPLO DE DATA WAREHOUSE OLAP (Desnormalizado - Modelo Estrella)")
print("="*80)
print("\n⭐ Tabla de HECHOS - Ventas (todo en una tabla para consultas rápidas):")
print(hechos_ventas)

print("\n📊 ANÁLISIS AGREGADO:")
print("-" * 80)

# Análisis típicos de OLAP
print("\n💰 Ingresos por Cliente:")
print(hechos_ventas.groupby('nombre_cliente')['ingreso_total'].sum().sort_values(ascending=False))

print("\n📦 Productos Más Vendidos:")
print(hechos_ventas.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False))

print("\n📅 Ventas por Fecha:")
print(hechos_ventas.groupby('fecha_pedido')['ingreso_total'].sum())

## 3. Comparación Detallada: OLTP vs OLAP

### Tabla Comparativa

In [ ]:
import pandas as pd

comparacion = pd.DataFrame({
    'Característica': [
        'Propósito',
        'Operaciones',
        'Tipo de consultas',
        'Volumen de transacciones',
        'Usuarios concurrentes',
        'Datos',
        'Diseño de BD',
        'Tiempo de respuesta',
        'Tamaño de BD',
        'Actualizaciones',
        'Ejemplo de consulta'
    ],
    'OLTP': [
        'Operaciones del día a día',
        'INSERT, UPDATE, DELETE',
        'Simples y rápidas',
        'Miles por segundo',
        'Cientos/Miles',
        'Actuales, detallados',
        'Normalizado (3FN)',
        'Milisegundos',
        'GB a cientos de GB',
        'Frecuentes (tiempo real)',
        'SELECT * FROM pedidos WHERE id=123'
    ],
    'OLAP': [
        'Análisis y toma de decisiones',
        'SELECT (lecturas)',
        'Complejas con agregaciones',
        'Pocas pero grandes',
        'Decenas',
        'Históricos, agregados',
        'Desnormalizado (Estrella)',
        'Segundos a minutos',
        'TB a PB',
        'Periódicas (batch, noche)',
        'SELECT cliente, SUM(ventas) ... GROUP BY ...'
    ]
})

print("\n" + "="*100)
print("COMPARACIÓN COMPLETA: OLTP vs OLAP")
print("="*100 + "\n")

# Imprimir tabla formateada
for idx, row in comparacion.iterrows():
    print(f"\n{row['Característica']}:")
    print(f"  🔵 OLTP: {row['OLTP']}")
    print(f"  🟢 OLAP: {row['OLAP']}")

## 4. Diferencias Arquitectónicas

### 4.1 Diseño de Base de Datos

#### OLTP - Normalizado (3FN)
```
Clientes ──┐
           ├──> Pedidos ───> Detalles_Pedido
Productos ─┘
```
**Ventajas:** Elimina redundancia, mantiene integridad

#### OLAP - Desnormalizado (Estrella)
```
      Dim_Cliente
            │
Dim_Tiempo─ Hechos_Ventas ─Dim_Producto
            │
      Dim_Ubicación
```
**Ventajas:** Consultas más rápidas, menos JOINs

In [ ]:
# Demostración: Diferencia de rendimiento en consultas
import time

print("\n" + "="*80)
print("COMPARACIÓN DE RENDIMIENTO: Consulta Analítica")
print("="*80)

# Consulta en OLTP (requiere múltiples JOINs)
print("\n🔵 OLTP - Normalizado (requiere 3 JOINs):")
start = time.time()
resultado_oltp = pedidos.merge(clientes, on='cliente_id') \
                        .merge(productos, on='producto_id')
resultado_oltp['total'] = resultado_oltp['cantidad'] * resultado_oltp['precio']
analisis_oltp = resultado_oltp.groupby('nombre_x')['total'].sum()
tiempo_oltp = time.time() - start
print(f"⏱️ Tiempo: {tiempo_oltp*1000:.2f} ms")
print(f"🔄 JOINs realizados: 3")

# Consulta en OLAP (todo en una tabla)
print("\n🟢 OLAP - Desnormalizado (sin JOINs):")
start = time.time()
analisis_olap = hechos_ventas.groupby('nombre_cliente')['ingreso_total'].sum()
tiempo_olap = time.time() - start
print(f"⏱️ Tiempo: {tiempo_olap*1000:.2f} ms")
print(f"🔄 JOINs realizados: 0")

print(f"\n⚡ OLAP es {(tiempo_oltp/tiempo_olap):.1f}x más rápido para consultas analíticas")
print("   (En datasets grandes, la diferencia es mucho mayor)")

## 5. ¿Cuándo Usar Cada Uno?

### Usa OLTP cuando:
- ✅ Necesitas procesar transacciones en tiempo real
- ✅ Las operaciones son principalmente INSERT/UPDATE/DELETE
- ✅ Requieres alta integridad y consistencia de datos
- ✅ Múltiples usuarios modifican datos simultáneamente
- ✅ Las consultas son simples y rápidas

### Usa OLAP cuando:
- ✅ Necesitas analizar grandes volúmenes de datos históricos
- ✅ Las operaciones son principalmente SELECT con agregaciones
- ✅ Requieres consultas complejas multidimensionales
- ✅ Los datos pueden ser actualizados en batch (no tiempo real)
- ✅ El objetivo es reporting y análisis, no transacciones

### Arquitectura Híbrida (Lo Más Común)

La mayoría de las organizaciones usan **ambos**:

```
OLTP (Sistema Transaccional)
    │
    │ ETL/ELT (noche)
    ↓
OLAP (Data Warehouse)
    ↓
Dashboards y Reportes
```

## 6. Ejercicio Práctico: Clasificación de Escenarios

Clasifica cada escenario como **OLTP** u **OLAP**:

In [ ]:
# Ejercicio interactivo
escenarios = [
    "1. Un cajero procesa una venta en una tienda física",
    "2. Un analista genera un reporte de ventas anuales por región",
    "3. Un cliente actualiza su dirección en el sitio web",
    "4. El gerente consulta las tendencias de ventas de los últimos 5 años",
    "5. Se registra un nuevo usuario en la aplicación móvil",
    "6. Se calcula el ROI de campañas de marketing del último trimestre",
    "7. Un sistema procesa 1000 transacciones bancarias por segundo",
    "8. Se identifica qué productos se venden juntos (market basket analysis)"
]

# Soluciones (descomenta para ver respuestas)
soluciones = [
    "OLTP - Transacción en tiempo real",
    "OLAP - Análisis histórico agregado",
    "OLTP - Actualización operacional",
    "OLAP - Consulta analítica de tendencias",
    "OLTP - Inserción de nuevo registro",
    "OLAP - Análisis de métricas de negocio",
    "OLTP - Alto volumen de transacciones",
    "OLAP - Data mining y patrones"
]

print("\n" + "="*80)
print("EJERCICIO: Clasifica cada escenario como OLTP u OLAP")
print("="*80 + "\n")

for escenario in escenarios:
    print(escenario)

print("\n" + "="*80)
print("Piensa en tu respuesta antes de ver las soluciones...")
print("="*80)

# Descomentar para ver soluciones:
# print("\n🔑 SOLUCIONES:\n")
# for i, (esc, sol) in enumerate(zip(escenarios, soluciones), 1):
#     print(f"{i}. {sol}")

## 7. Resumen y Puntos Clave

### OLTP (Transaccional)
- 🎯 **Propósito:** Operaciones del día a día
- 💾 **Diseño:** Normalizado (evita redundancia)
- ⚡ **Operaciones:** INSERT, UPDATE, DELETE
- 👥 **Usuarios:** Miles concurrentes
- ⏱️ **Respuesta:** Milisegundos

### OLAP (Analítico)
- 🎯 **Propósito:** Análisis y toma de decisiones
- 💾 **Diseño:** Desnormalizado (optimiza consultas)
- ⚡ **Operaciones:** SELECT con agregaciones
- 👥 **Usuarios:** Decenas de analistas
- ⏱️ **Respuesta:** Segundos a minutos

### Arquitectura Moderna
La mayoría de las organizaciones necesitan **AMBOS** sistemas trabajando en conjunto:
- **OLTP** para operaciones en tiempo real
- **OLAP** para análisis y reporting
- **ETL/ELT** para sincronizar datos entre ambos

## Próximos Pasos

En el siguiente notebook exploraremos los **fundamentos del modelado de datos**, incluyendo metadatos y calidad del dato.

---

## Referencias

- Codd, E. F., Codd, S. B., & Salley, C. T. (1993). *Providing OLAP to User-Analysts*
- Kimball, R. (1996). *The Data Warehouse Toolkit*
- Inmon, W. H. (2005). *Building the Data Warehouse*

## 15. Actualización 2025: OLTP vs OLAP y Convergencia

### Evolución y Tendencias
- Separación clara de workloads sigue vigente (OLTP transaccional vs OLAP analítico) por aislamiento de patrones de acceso y gobernanza.
- CDC (Change Data Capture) se consolida como puente estándar: Debezium (open source) ampliamente adoptado; proveedores gestionados (Fivetran, Airbyte) reducen fricción.
- HTAP (Hybrid Transactional/Analytical Processing) emerge pero selectivo: SingleStore, TiDB, AlloyDB "lakehouse acceleration"; útil en casos de baja latencia analítica (segundos) + volumen medio.
- Real-Time Analytics stores: Apache Pinot, ClickHouse, Druid para dashboards sub-segundo; complementan almacenes columnar (BigQuery, Snowflake) y formatos abiertos (Delta/Iceberg/Hudi).
- Materialized Views incrementales + streaming (Flink, Kafka Streams) reemplazan ETLs batch nocturnos en escenarios críticos.

### Buenas Prácticas 2025 (No Alucinaciones)
| Tema | Práctica Recomendada | Herramientas Maduros |
|------|----------------------|----------------------|
| CDC | Log-based, idempotente, esquema versionado | Debezium, MySQL/Postgres binlog, Oracle GoldenGate |
| Formatos OLAP | Columnar abierto + ACID | Parquet + Delta / Iceberg / Hudi |
| Latencia | Micro-batch (1–5 min) antes de real-time si costo streaming alto | Spark Structured Streaming, Flink |
| Consistencia | Contratos de datos + validación en ingestión | Great Expectations, Soda, dbt tests |
| Gobernanza | Catálogo + lineage automático | DataHub, OpenLineage |
| Costos | Tiering almacenamiento (hot/warm/cold) | S3 + Glacier, GCS Nearline |

### Cuándo Considerar HTAP
- Lecturas analíticas ligeras (agregaciones simples) sobre datos muy recientes (<5s) sin necesidad de transformar.
- Equipo con necesidad de reducir copia de datos y complejidad pipelines.
- Trade-off: mayor complejidad operativa, potencial contención recursos.

### Patrón Recomendado General
1. OLTP limpio (PostgreSQL/MySQL) con claves claras + restricciones.
2. CDC → Cola (Kafka) con esquema avro/protobuf registrado (Schema Registry).
3. Procesamiento incremental (Flink/Spark) enriqueciendo y normalizando.
4. Capa Lakehouse (Delta/Iceberg) para histórico consolidado.
5. Servicio Real-Time (Pinot/ClickHouse) para métricas sub-segundo específicas.

## 16. Mini-Ejercicio: Modelado Fact & Dimensión con Contrato

### Escenario
Sistema de ventas genera tablas transaccionales:
- `orders(order_id, customer_id, order_ts, status, total_amount)`
- `order_items(item_id, order_id, product_id, quantity, unit_price)`

Objetivo: Diseñar modelo analítico inicial (OLAP) con una tabla de hechos y dimensiones necesarias, más un contrato de datos mínimo para la fact table.

### Tareas
1. Definir `dim_customer` (llave, atributos clave, justificación de exclusión de campos poco usados).
2. Definir `dim_product` (atributos vs normalización adicional; decidir si separar categoría en otra dimensión).
3. Diseñar `fact_order_item` indicando granularidad (nivel item) y justificar por qué no nivel order.
4. Especificar 5 métricas derivadas (revenue, avg_order_value, items_per_order, gross_margin si existiera, return_rate) con definición y dependencia.
5. Redactar contrato de datos simplificado (tabla) para `fact_order_item` con: campo, tipo lógico, regla calidad, severidad.
6. Identificar 3 riesgos de diseño (ej. cambios en esquema OLTP, llegadas tardías, SCD en clientes) y mitigación.

### Formato Esperado
- Tabla dimensiones: nombre, llave, atributos, justificación.
- Tabla métricas: nombre, fórmula, nivel agregación.
- Contrato datos: campo, tipo, regla (expresión), severidad.

### Rúbrica
| Aspecto | Peso |
|---------|------|
| Granularidad y justificación | 30% |
| Definición dimensiones clara | 25% |
| Métricas bien definidas | 20% |
| Contrato datos (reglas medibles) | 15% |
| Riesgos y mitigaciones | 10% |

### Siguientes Pasos (Opcional)
- Preparar script de validación SQL (COUNT DISTINCT vs duplicados, null checks).
- Incluir surrogate keys para dimensiones y discutir manejo de SCD Tipo 2.

> Este ejercicio prepara base para notebooks de modelado dimensional más profundo.


In [ ]:
# Plantilla Python opcional para contrato de datos (estudiante puede adaptar)
contract_fact_order_item = [
    {"field": "order_id", "type": "string", "rule": "not null and unique within fact", "severity": "Alta"},
    {"field": "item_id", "type": "string", "rule": "not null", "severity": "Alta"},
    {"field": "product_id", "type": "string", "rule": "not null", "severity": "Alta"},
    {"field": "quantity", "type": "integer", "rule": "quantity > 0", "severity": "Alta"},
    {"field": "unit_price", "type": "decimal", "rule": "unit_price >= 0", "severity": "Media"},
    {"field": "total_amount", "type": "decimal", "rule": "total_amount == quantity * unit_price", "severity": "Media"},
]

for c in contract_fact_order_item:
    print(c)

# Placeholder para cálculo métricas (estudiante luego implementa con DataFrame)
# def compute_metrics(df_fact):
#     return {
#         'revenue': (df_fact['total_amount']).sum(),
#         'avg_order_value': df_fact.groupby('order_id')['total_amount'].sum().mean(),
#         'items_per_order': df_fact.groupby('order_id')['item_id'].count().mean(),
#         'return_rate': 'requires status/returns info',
#     }
